In [1]:
from unsloth import FastVisionModel
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3-VL-2B-Instruct-bnb-4bit",
    load_in_4bit = True,
    use_gradient_checkpointing = "unsloth"
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.11.3: Fast Qwen3_Vl patching. Transformers: 4.57.1.
   \\   /|    NVIDIA GeForce RTX 4070 SUPER. Num GPUs = 1. Max memory: 11.994 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [2]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers

    r = 16,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [3]:
from datasets import load_dataset
dataset = load_dataset("invoices_dataset/mini_dataset/images", split="train")

In [4]:
instruction = "Read the OCR in the image."

def convert_to_conversation(sample):
    conversation = [
        { "role": "user",
          "content" : [
            {"type" : "text",  "text"  : instruction},
            {"type" : "image", "image" : sample["image"]} ]
        },
        { "role" : "assistant",
          "content" : [
            {"type" : "text",  "text"  : sample["invoice_nr"]} ]
        },
    ]
    return { "messages" : conversation }
pass

In [5]:
converted_dataset = [convert_to_conversation(sample) for sample in dataset]

In [6]:
FastVisionModel.for_inference(model)

image = dataset[2]["image"]
instruction = "Read the OCR in the image."

messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": instruction}
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 400,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

Invoice no: 62517865
Date of issue: 06/02/2015

Seller:
Trujillo-Hunt
430 Mark Ferry Suite 495
Maxside, DC 65686
Tax Id: 992-71-8540
IBAN: GB73HCPH34959888432899

Client:
Lee and Sons
8552 Karen Islands
East Roger, ID 40416
Tax Id: 929-86-0601

ITEMS
No.	Description	Qty	UM	Net price	Net worth	VAT [%]	Gross worth
1.	Care & Repair of Furniture	2,00	each	4,25	8,50	10%	9,35

SUMMARY
VAT [%]	Net worth	VAT	Gross worth
10%	8,50	0,85	9,35
Total		$ 8,50	$ 0,85	$ 9,35<|im_end|>


In [7]:
import mlflow
import os

cwd = os.getcwd()
if os.access(cwd, os.W_OK):
    db_path = os.path.abspath("mlflow.db")
else:
    home_mlflow_dir = os.path.expanduser("~/mlflow_local")
    os.makedirs(home_mlflow_dir, exist_ok=True)
    db_path = os.path.join(home_mlflow_dir, "mlflow.db")

os.makedirs(os.path.dirname(db_path), exist_ok=True)

mlflow.set_tracking_uri(f"sqlite:///{db_path}")
mlflow.set_experiment("qwen3-vl-invoice-finetune")

2025/11/25 20:11:04 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/11/25 20:11:04 INFO mlflow.store.db.utils: Updating database tables
2025-11-25 20:11:04 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
2025-11-25 20:11:04 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
2025-11-25 20:11:04 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
2025-11-25 20:11:04 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


<Experiment: artifact_location='/home/isac/AAIS/AAIS_project/mlruns/2', creation_time=1764087527768, experiment_id='2', last_update_time=1764087527768, lifecycle_stage='active', name='qwen3-vl-invoice-finetune', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [8]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
from datetime import datetime

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = converted_dataset,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 30,
        # num_train_epochs = 1, # Set this instead of max_steps for full training runs
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "mlflow",
        run_name=f"qwen3-vl-invoice-finetune-{datetime.now().strftime('%Y%m%d-%H%M%S')}",

        # Below items are required for vision finetuning:
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 2048,
    ),
)

Unsloth: Model does not have a default image size - using 512


In [9]:
trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3 | Num Epochs = 30 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 23,724,032 of 2,151,256,064 (1.10% trained)


Step,Training Loss
1,3.976000
2,3.976000
3,3.719700
4,3.222000
5,2.643400
6,2.153100
7,1.834900
8,1.572200
9,1.342000
10,1.159700


Unsloth: Will smartly offload gradients to save VRAM!


TrainOutput(global_step=30, training_loss=1.049771230792006, metrics={'train_runtime': 47.658, 'train_samples_per_second': 5.036, 'train_steps_per_second': 0.629, 'total_flos': 391988201472000.0, 'train_loss': 1.049771230792006, 'epoch': 30.0})

In [10]:
FastVisionModel.for_inference(model)

image = dataset[2]["image"]
instruction = "Read the OCR in the image."

messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": instruction}
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

62517865<|im_end|>


In [11]:
last_run_id = mlflow.last_active_run().info.run_id

with mlflow.start_run(run_id=last_run_id):
    mlflow.log_param("model_name", "unsloth/Qwen3-VL-2B-Instruct-bnb-4bit")
    mlflow.log_param("finetune_task", "invoice_number_extraction")
    mlflow.log_params(model.peft_config)

    # Path where SFTTrainer saved the checkpoint/adapter
    adapter_path = "outputs/checkpoint-30"
    assert os.path.isdir(adapter_path), f"Adapter folder not found: {adapter_path}"

    import mlflow.pyfunc

    class PEFTVisionWrapper(mlflow.pyfunc.PythonModel):
        def load_context(self, context):
            adapter_local = context.artifacts["adapter"]

            from unsloth import FastVisionModel
            from peft import PeftModel

            # Load base model & tokenizer
            base_name = "unsloth/Qwen3-VL-2B-Instruct-bnb-4bit"
            model, tokenizer = FastVisionModel.from_pretrained(
                base_name,
                load_in_4bit=True,
                use_gradient_checkpointing="unsloth",
            )

            # Attach the saved PEFT adapters
            model = PeftModel.from_pretrained(model, adapter_local)

            self.model = model.eval()
            self.tokenizer = tokenizer

        def predict(self, context, model_input):
            import torch
            from PIL import Image

            results = []
            for _, row in model_input.iterrows():
                img = row["image"]
                if isinstance(img, str):
                    img = Image.open(img).convert("RGB")
                instruction = row.get("instruction", "Read the OCR in the image.")

                messages = [
                    {"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": instruction}]}
                ]
                input_text = self.tokenizer.apply_chat_template(messages, add_generation_prompt=True)
                inputs = self.tokenizer(img, input_text, return_tensors="pt").to(next(self.model.parameters()).device)
                with torch.no_grad():
                    gen = self.model.generate(**inputs, max_new_tokens=400)
                decoded = self.tokenizer.batch_decode(gen, skip_special_tokens=True)
                results.append(decoded[0] if isinstance(decoded, (list, tuple)) else str(decoded))
            import pandas as pd
            return pd.DataFrame({"prediction": results})

    mlflow.pyfunc.log_model(
        python_model=PEFTVisionWrapper(),
        name="qwen3vl_finetuned_extraction",
        artifacts={"adapter": adapter_path}
    )

/home/isac/miniconda3/envs/test_env/lib/python3.13/site-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


2025/11/25 20:12:30 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
